In [ ]:
%load_ext autoreload
%autoreload 2

### Set up

In [ ]:
"""
This cell does the initial project setup.
A script with this code uses the location of the `.env` file as the anchor for
the whole project (= PROJECT_ROOT). Add `src` directory to sys path.
"""
from pathlib import Path
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))
print(f"Project root directory: {PROJECT_ROOT}")

# Prefer a local connectome_interpreter checkout (CI_DEV_PATH in .env) over the
# pixi-installed wheel, so edits in the fork take effect via %autoreload 2.
# insert(0, ...) is required: it must precede site-packages in the search order.
import os
ci_dev_path = os.environ.get('CI_DEV_PATH')
if ci_dev_path and Path(ci_dev_path).is_dir():
    sys.path.insert(0, str(Path(ci_dev_path)))
    print(f"connectome_interpreter dev clone: {ci_dev_path}")
else:
    print("connectome_interpreter: using the pixi-installed package")

In [ ]:
from utils import olc_client
c = olc_client.connect(verbose=True)

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp

import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

import navis
import neuprint
from neuprint import fetch_adjacencies
import connectome_interpreter as ci

# Guard against silently falling back to the pixi-installed copy.
print(f"connectome_interpreter loaded from: {ci.__file__}")

In [ ]:
# import sys
# import os
# from pathlib import Path
# PROJECT_ROOT = Path(os.path.dirname(os.getcwd()))
# sys.path.append(str(PROJECT_ROOT.joinpath('src')))

# from connectome_interpreter import *

In [ ]:
dataset = 'malecns_v1.0'
# dataset = 'fafb_all_neurons'

data_folder = Path(PROJECT_ROOT, 'cache', 'data')

results_folder = PROJECT_ROOT / 'results' / dataset
results_folder.mkdir(parents=True, exist_ok=True)

# Getting data

`meta` contains information about neurons. <br>
`conn_mat` is direct connectivity, without signs.

Optional: remove connections between specified cell types

## Neuron information

In [ ]:
# get meta
# the idx column records the index of the neuron in inprop
meta = pd.read_csv(data_folder / f'{dataset}_meta.csv', index_col = 0).reset_index()
meta

In [ ]:
# simplify neurotransmitter types and define colors
meta['nt'] = meta['nt'].replace(['histamine', 'dopamine', 'octopamine', 'serotonin'], 'other')          
nt_to_color = {'acetylcholine': '#EE672D', 'glutamate': '#09A64D', 'gaba': '#1F4695', 'other': '#979DA5', 'unclear': '#979DA5'}

#signs from neurotransmitter types
meta['sign'] = meta['nt'].replace({'acetylcholine': 1, 'glutamate': -1, 'gaba': -1, 'other': 1, 'unclear': 1})
#make L1 excitatory
idx_L1 = meta[meta.cell_type == 'L1'].idx.tolist()
meta.loc[idx_L1, 'sign'] = 1

In [ ]:
# make dictionaries that map indices to meta info
idx_to_type = dict(zip(meta.idx, meta.cell_type))
idx_to_instance = dict(zip(meta.idx, meta.instance))
idx_to_coords = dict(zip(meta.idx, meta.coords))
idx_to_root = dict(zip(meta.idx, meta.bodyId))
idx_to_sign = dict(zip(meta.idx, meta.sign))

In [ ]:
# the cell types one can get hex coordinates for
coord_types = meta[~meta.coords.isna()].instance.unique()
coord_types

## Propagated connectivity

In [ ]:
# connectivity is a sparse matrix
# pre in rows, post in columns
conn_mat = sp.sparse.load_npz(data_folder / f'{dataset}_prop.npz') #full integer weights

# normalize to get input-normalized weights
total_post = conn_mat.sum(axis=0).A1.astype(float)
col_sums_with_inversion = np.reciprocal(
    total_post, where=total_post != 0
)
conn_mat = conn_mat.multiply(col_sums_with_inversion)
conn_mat = conn_mat.astype(np.float32)

## Optional: remove certain connections
Remove weights between all cell types that start with strings defined by type_del_pre and type_del_post

In [ ]:
# type_del_pre = 'Mi1'
# type_del_post = 'T4a'

# idx_pre = np.array([meta.loc[i,'idx'] for i in meta.index if meta.loc[i,'cell_type'].startswith(type_del_pre)])
# idx_post = np.array([meta.loc[i,'idx'] for i in meta.index if meta.loc[i,'cell_type'].startswith(type_del_post)])

# conn_mat = modify_coo_matrix(conn_mat, idx_pre, idx_post, 0, re_normalize=False)
# conn_mat = conn_mat.tocsc()

# Analyze input-output relationship between cell types of interest
Manually specify the following: <br>
`input_types` is a list of cell types which we treat as inputs. <br>
`output_types` is a list of cell types which we treat as outputs. <br>
`nr_hops_max` is the maximum number of hops (steps) allowed in the path search between input and output neurons. <br>
`direct_thre` is a lower threshold on the direct input-normalized connectivity. If 0 then all connections are considered. <br>
`signed` indicates whether to use signs (True) or not (False)

Notes: <br>
`inidx` and `outidx` below we also constrain the `input_types` and `output_types` to be on the right side only. <br>
`combining_method` below takes the mean connectivity over post neurons but is summed over pre neurons. <br>
Setting `nr_hops_max` too large (>4) could be slow. Check the plot of the summed effective input weight below. <br>
If `signed=True` then GABA and glutamate are treated as inhibitory and everything else as excitatory.

In [ ]:
# input_types = ['L1', 'L2', 'L3', 'R8', 'R8d', 'R7', 'R7d', 'HBeyelet']
input_types = ['L1', 'L2', 'L3', 'R8', 'R8d', 'R7', 'R7d']

# output_types = ['T4a', 'T4b', 'T4c', 'T4d']
output_types = ['LC10a']

nr_hops_max = 4

direct_thre = 0.001

signed = True

In [ ]:
# find connectivity indices and print how many neurons
inidx = meta.idx[meta.cell_type.isin(input_types) & (meta.side=='right')]
print('Number of {} neurons: {}'.format(input_types, inidx.shape[0]))
outidx = meta.idx[meta.cell_type.isin(output_types) & (meta.side=='right')]
print('Number of {} neurons: {}'.format(output_types, outidx.shape[0]))

In [ ]:
# summed effective input weights per number of hops
path_lengths = list(range(1, nr_hops_max + 1))
summed_weights = []

for i in path_lengths:
    paths = ci.find_paths_of_length(conn_mat, inidx, outidx, i)
    if (paths is None) or (paths.empty):
        summed_weights.append(0)
        continue

    paths = ci.filter_paths(paths, direct_thre)
    if (paths is None) or (paths.empty):
        summed_weights.append(0)
        continue

    if not signed:
        tmp_df = ci.effective_conn_from_paths(paths, wide=False)
        tmp_df = ci.group_paths(tmp_df, idx_to_type, idx_to_type, combining_method='mean')
        tmp_df = tmp_df.pivot(index='pre', columns='post', values='weight').fillna(0)
    else:
        tmp_df_e, tmp_df_i = ci.signed_effective_conn_from_paths(paths, wide=False, idx_to_nt=idx_to_sign)
        tmp_df_e = ci.group_paths(tmp_df_e, idx_to_type, idx_to_type, combining_method='mean')
        tmp_df_i = ci.group_paths(tmp_df_i, idx_to_type, idx_to_type, combining_method='mean')
        tmp_df_e = tmp_df_e.pivot(index='pre', columns='post', values='weight').fillna(0)
        tmp_df_i = tmp_df_i.pivot(index='pre', columns='post', values='weight').fillna(0)
        tmp_df = tmp_df_e.sub(tmp_df_i, fill_value=0)

    summed_weights.append(tmp_df.abs().values.sum())

fig = px.line(x=path_lengths, y=summed_weights, markers=True)
fig.update_xaxes(range=[0.9, nr_hops_max+.1], title='number of hops')
fig.update_yaxes(title='summed effective input weight')
fig.show()


### pie charts of relative effective weight from input cell types

`input_colors` is a dictionary associating colors with each element of `input_types`.

In [ ]:
input_colors = {'L1': '#b4e5cb', 'L2': '#029356', 'L3': '#8fce0a',\
            'R7': "#8e56f5", 'R8': "#51bfc7", 'HBeyelet': '#151515',\
            'R7d': "#4220B4", 'R8d': "#099C9C"}

type_to_sign = dict(zip(meta.cell_type, meta.sign))   # add near the other dicts


In [ ]:
df = pd.DataFrame(np.zeros((len(input_types), len(output_types))), index=input_types, columns=output_types)
for i in range(1, nr_hops_max + 1):
    paths = ci.find_paths_of_length(conn_mat, inidx, outidx, i)
    if (paths is None) or (paths.empty):
        continue
    paths = ci.filter_paths(paths, direct_thre)
    if (paths is None) or (paths.empty):
        continue

    if not signed:
        tmp_df = ci.effective_conn_from_paths(paths, wide=False)
        tmp_df = ci.group_paths(tmp_df, idx_to_type, idx_to_type, combining_method='mean')
        tmp_df = tmp_df.pivot(index='pre', columns='post', values='weight').fillna(0)
    else:
        tmp_df_e, tmp_df_i = ci.signed_effective_conn_from_paths(paths, wide=False, idx_to_nt=idx_to_sign)
        tmp_df_e = ci.group_paths(tmp_df_e, idx_to_type, idx_to_type, combining_method='mean')
        tmp_df_i = ci.group_paths(tmp_df_i, idx_to_type, idx_to_type, combining_method='mean')
        tmp_df_e = tmp_df_e.pivot(index='pre', columns='post', values='weight').fillna(0)
        tmp_df_i = tmp_df_i.pivot(index='pre', columns='post', values='weight').fillna(0)

        tmp_df = tmp_df_e.sub(tmp_df_i, fill_value=0)
    df.loc[tmp_df.index, tmp_df.columns] += np.abs(tmp_df)
    
if df.sum().max() > 0:
    for output_type in output_types:
        plot_df = df.loc[:, output_type].to_frame().stack(0).reset_index(name='effective_weight').rename(
            columns={'level_1':'cell_type', 'level_0':'input_type'})

        fig = px.pie(plot_df, values='effective_weight', names='input_type', 
                    title=f"{output_type}, tot. effective weight: {plot_df.effective_weight.sum():.4f}",
                    color='input_type', color_discrete_map=input_colors)
        fig.update_traces(textposition='inside', texttemplate='%{percent:.0%}')
        fig.update_layout(
            width=350,
            height=350,
            plot_bgcolor = 'white',
            paper_bgcolor = 'white',
            font={
                "family":'Arial'
                , "size" : 14
            }
        )
        fig.show()
else:
    print('No effective upstream connections found for input types to output types.')

### barplot of effective weight from input cell types
Same numbers as the pies above, but a bar chart lets you compare magnitudes across
output types on a shared axis (a pie only shows within-target proportions).
`value_col='effective_weight'` for absolute weights, `'fraction'` for per-target shares.


In [ ]:
def plot_input_weight_bars(
    weight_df: pd.DataFrame,
    input_types: list[str],
    output_types: list[str],
    input_colors: dict[str, str],
    value_col: str = 'effective_weight',
    title: str = 'effective weight from input cell types',
) -> go.Figure:
    """Grouped barplot of effective input weight per input cell type, faceted by output type.

    Args:
        weight_df: input types in rows, output types in columns (as built in the pie cell).
        value_col: 'effective_weight' for absolute values, 'fraction' for the
            within-output-type share (the quantity the pies show).
    """
    missing_colors = [t for t in input_types if t not in input_colors]
    if missing_colors:
        print(f'WARNING: no color assigned for {missing_colors}; plotly defaults will be used')

    plot_df = (
        weight_df.reindex(index=input_types, columns=output_types)
        .fillna(0)
        .rename_axis(index='input_type')
        .reset_index()
        .melt(id_vars='input_type', var_name='output_type', value_name='effective_weight')
    )
    totals = plot_df.groupby('output_type')['effective_weight'].transform('sum')
    plot_df['fraction'] = plot_df.effective_weight.div(totals).fillna(0)

    fig = px.bar(
        plot_df,
        x='input_type',
        y=value_col,
        color='input_type',
        color_discrete_map=input_colors,
        category_orders={'input_type': input_types, 'output_type': output_types},
        facet_col='output_type' if len(output_types) > 1 else None,
        text=plot_df.fraction.map('{:.0%}'.format),
        title=title,
    )
    # x-axis already names each bar, so the legend would be redundant
    fig.update_traces(textposition='outside', cliponaxis=False, marker_line_width=0)
    fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig.update_xaxes(title='', showgrid=False)
    fig.update_yaxes(
        title='effective weight' if value_col == 'effective_weight' else 'fraction of total',
        showgrid=True, gridcolor='#EEEEEE', zeroline=True, zerolinecolor='#BBBBBB',
    )
    fig.update_layout(
        showlegend=False,
        width=180 + 55 * len(input_types) * max(len(output_types), 1),
        height=400,
        plot_bgcolor='white',
        paper_bgcolor='white',
        bargap=0.25,
        font={'family': 'Arial', 'size': 14},
    )
    return fig


if df.sum().max() > 0:
    plot_input_weight_bars(df, input_types, output_types, input_colors).show()
else:
    print('No effective upstream connections found for input types to output types.')


### anatomical receptive field (ARF) predictions
ARFs are summed over all input bodyIds and input cell types <br>
`hex_heatmap`: effective weights in ME column coordinates <br>
`plot_mollweide_projection`: effective weights in visual space (after eyemap)

In [ ]:
#pick the output bodyIds of which the RF shall be shown (change by slider)
rfidx = outidx[:3]

In [ ]:
all_coords = meta['coords'].unique()
df = pd.DataFrame(
    np.zeros((len(all_coords), len(rfidx))), 
    index=all_coords, 
    columns=rfidx.map(idx_to_root).values)
for i in range(1, nr_hops_max + 1):
    paths = ci.find_paths_of_length(conn_mat, inidx, rfidx, i)
    if (paths is None) or paths.empty:
        continue
    paths = ci.filter_paths(paths, direct_thre)
    if (paths is None) or (paths.empty):
        continue

    # grouping (relabelling pre -> coords, post -> rootId) is no longer done inside
    # effective_conn_from_paths()/signed_effective_conn_from_paths(); call group_paths()
    # explicitly on the long-format output, then pivot to wide ourselves.
    if not signed:
        tmp_df = ci.effective_conn_from_paths(paths, wide=False)
        tmp_df = ci.group_paths(tmp_df, idx_to_coords, idx_to_root, combining_method='mean')
        tmp_df = tmp_df.pivot(index='pre', columns='post', values='weight').fillna(0)
    else:
        tmp_df_e, tmp_df_i = ci.signed_effective_conn_from_paths(paths, wide=False, idx_to_nt=idx_to_sign)
        tmp_df_e = ci.group_paths(tmp_df_e, idx_to_coords, idx_to_root, combining_method='mean')
        tmp_df_i = ci.group_paths(tmp_df_i, idx_to_coords, idx_to_root, combining_method='mean')
        tmp_df_e = tmp_df_e.pivot(index='pre', columns='post', values='weight').fillna(0)
        tmp_df_i = tmp_df_i.pivot(index='pre', columns='post', values='weight').fillna(0)
        tmp_df = tmp_df_e.sub(tmp_df_i, fill_value=0)
    tmp_df = tmp_df[(tmp_df.index != "nan") & (~tmp_df.index.isnull())] #remove inputs without column assignment
    tmp_df.columns = tmp_df.columns.astype(int)
    df.loc[tmp_df.index, tmp_df.columns] += tmp_df

In [ ]:
fig = ci.hex_heatmap(df,
                eyemap_path=Path(PROJECT_ROOT, 'params', 'eyemaps', 'eyemap_mcns_f20240701','right.xlsx'),
                  custom_colorscale='RdBu_r', 
                  global_min=-np.abs(df.values).max(), 
                  global_max=np.abs(df.values).max())
fig.show()

In [ ]:
fig = ci.plot_mollweide_projection(
    df,
    eyemap_path=Path(PROJECT_ROOT, 'params', 'eyemaps', 'eyemap_mcns_f20240701','right.xlsx'),
    # eyemap_path=str(eyemap_path),
    custom_colorscale='RdBu_r', 
    global_min=-np.abs(df.values).max(), 
    global_max=np.abs(df.values).max())
fig.show()

# Effective contribution via the summed propagation matrix

The sections above enumerate paths per hop count (`find_paths_of_length`) and then
reduce them. The alternative used in `github_ol_analysis` propagates **once** —
`stepsn = A + A² + … + A^N` for `A = conn_mat` — saves that matrix, and reads
effective weights straight out of it. One dense slice per query instead of a path
enumeration, so it is the cheaper route when you query many input/output sets.

Assumes the summed matrix is already on disk (see the commented builder cell).

**Caveats, in order of how much they bite:**
1. *Forward-flow trimming.* `github_ol_analysis` trims `inprop` to forward-directed
   edges by hitting time (`setup_data/flow.py:trim_inprop_by_flow`) **before**
   propagating. Untrimmed, `Aⁿ` accumulates recurrent loops and the weights stop
   being interpretable as feedforward contribution. If the local matrix was not
   trimmed, its numbers are not comparable to the path-based ones above.
2. *Signs.* A summed matrix built from the unsigned `conn_mat` is all-positive.
   Signed requires `compress_paths_signed`, which returns separate excitatory and
   inhibitory stacks; sum each and subtract.
3. *Hop-count aggregation differs.* The pie/bar cells sum |weight| per hop count.
   Here signs and hop counts cancel *before* you take the absolute value, so the
   totals will not match cell #27 even with identical inputs.
4. *No `direct_thre` equivalent.* Path filtering happens inside `compress_paths` as
   `output_threshold`, applied per step at build time — not at query time.


In [ ]:
n_steps_sum = 4  # matches nr_hops_max above

# unsigned: one file. signed: an (excitatory, inhibitory) pair, as compress_paths_signed returns.
sum_path = data_folder / f'{dataset}_stepsn_sum_{n_steps_sum}step.npz'
sum_path_e = data_folder / f'{dataset}_stepsn_sum_{n_steps_sum}step_e.npz'
sum_path_i = data_folder / f'{dataset}_stepsn_sum_{n_steps_sum}step_i.npz'

if signed and sum_path_e.exists() and sum_path_i.exists():
    stepsn = (sp.sparse.load_npz(sum_path_e) - sp.sparse.load_npz(sum_path_i)).tocsr()
    stepsn_is_signed = True
elif sum_path.exists():
    stepsn = sp.sparse.load_npz(sum_path).tocsr()
    stepsn_is_signed = False
    if signed:
        print('WARNING: signed=True but only the unsigned summed matrix is on disk; '
              'results below are magnitudes.')
else:
    raise FileNotFoundError(
        f'No summed propagation matrix found. Expected one of:\n'
        f'  {sum_path}\n  {sum_path_e} + {sum_path_i}\n'
        f'Run the builder cell below (heavy) or copy the files over.'
    )

print(f'stepsn: {stepsn.shape} nnz={stepsn.nnz:,}  signed={stepsn_is_signed}')


In [ ]:
# # Recycled from github_ol_analysis/src/setup_data/flow.py:compute_lat_flow_sum
# # NOTE: for comparability with that repo, trim conn_mat to forward-directed edges
# # by hitting time first (flow.py:trim_inprop_by_flow) — not done here.
#
# steps = ci.compress_paths(
#     conn_mat, step_number=n_steps_sum, output_threshold=1e-6,
#     chunkSize=5000, save_to_disk=False, return_results=True,
# )
# sp.sparse.save_npz(sum_path, ci.add_first_n_matrices(steps, len(steps)).tocsc())
#
# # signed variant: even numbers of inhibition count as excitation
# steps_e, steps_i = ci.compress_paths_signed(
#     conn_mat, idx_to_sign, target_layer_number=n_steps_sum,
#     output_threshold=1e-6, chunkSize=2000, save_to_disk=False, return_results=True,
# )
# sp.sparse.save_npz(sum_path_e, ci.add_first_n_matrices(steps_e, len(steps_e)).tocsc())
# sp.sparse.save_npz(sum_path_i, ci.add_first_n_matrices(steps_i, len(steps_i)).tocsc())


In [ ]:
def grouped_effweight(
    stepsn,
    inidx,
    outidx,
    inidx_map: dict,
    outidx_map: dict,
    combining_method: str = 'mean',
) -> pd.DataFrame:
    """Grouped effective weight read out of a summed propagation matrix.

    Sums the effective weight over input neurons within each `inidx_map` group, then
    combines ('median'/'mean'/'sum') over output neurons within each `outidx_map`
    group. Returns rows = output groups, cols = input groups, with no thresholding,
    sorting or printing. Unlike `ci.result_summary` it never averages over group
    members absent from `inidx`/`outidx`.

    Recycled from github_ol_analysis/src/utils/core_data.py:285.
    """
    if combining_method not in ('median', 'mean', 'sum'):
        raise ValueError(
            f"combining_method must be 'median', 'mean' or 'sum'; got {combining_method!r}"
        )
    inidx = np.asarray(inidx)
    outidx = np.asarray(outidx)
    # slice rows first: cheap on CSR, and shrinks the matrix before the expensive
    # column indexing (columns-first scans all nonzeros)
    in_out_df = pd.DataFrame(
        data=stepsn[inidx, :][:, outidx].toarray(),
        index=[str(inidx_map[k]) for k in inidx],
        columns=outidx,
    )
    in_out_df = in_out_df.groupby(in_out_df.index).sum()
    in_out_df.columns = [str(outidx_map[k]) for k in in_out_df.columns]
    grouped = in_out_df.T.groupby(level=0)
    return getattr(grouped, combining_method)()


In [ ]:
# rows = output groups, cols = input groups -> transpose to match the `df` above
eff_sum = grouped_effweight(
    stepsn, inidx, outidx, idx_to_type, idx_to_type, combining_method='mean',
).T
eff_sum = eff_sum.reindex(index=input_types, columns=output_types).fillna(0)
eff_sum


In [ ]:
plot_input_weight_bars(
    eff_sum.abs(), input_types, output_types, input_colors,
    title=f'effective weight from input cell types ({n_steps_sum}-step summed matrix)',
).show()

for output_type in output_types:
    plot_df = (
        eff_sum[output_type].abs().rename('effective_weight')
        .rename_axis('input_type').reset_index()
    )
    fig = px.pie(
        plot_df, values='effective_weight', names='input_type',
        title=f"{output_type}, tot. effective weight: {plot_df.effective_weight.sum():.4f}",
        color='input_type', color_discrete_map=input_colors,
    )
    fig.update_traces(textposition='inside', texttemplate='%{percent:.0%}')
    fig.update_layout(
        width=350, height=350, plot_bgcolor='white', paper_bgcolor='white',
        font={'family': 'Arial', 'size': 14},
    )
    fig.show()


In [ ]:
# coords in rows, bodyId in columns. outidx_map=idx_to_root makes each output body its
# own group, so combining_method is an identity there.
rf_sum = grouped_effweight(
    stepsn, inidx, rfidx, idx_to_coords, idx_to_root, combining_method='mean',
).T
# drop inputs without a column assignment
rf_sum = rf_sum[(rf_sum.index != 'nan') & (rf_sum.index != 'None') & (~rf_sum.index.isnull())]
# .astype('int64'), not astype(int): the latter truncates large bodyIds to int32 on Windows
rf_sum.columns = rf_sum.columns.astype('int64')
print(f'{rf_sum.shape[0]} columns x {rf_sum.shape[1]} target neurons')
rf_sum.head()


In [ ]:
eyemap_path = Path(PROJECT_ROOT, 'params', 'eyemaps', 'eyemap_mcns_f20240701', 'right.xlsx')
rf_max = np.abs(rf_sum.values).max()

for plot_fn in (ci.hex_heatmap, ci.plot_mollweide_projection):
    plot_fn(
        rf_sum,
        eyemap_path=eyemap_path,
        custom_colorscale='RdBu_r',
        global_min=-rf_max,
        global_max=rf_max,
    ).show()
